# Public Private-Credit Market Analysis

This notebook builds the figures used to examine public business development companies (BDCs), broader risk assets, high-yield credit spreads, drawdowns, volatility, and the relative performance of FSK against a peer basket.

Price data are downloaded from Yahoo Finance through `yfinance`. High-yield option-adjusted spread data come from the Federal Reserve Bank of St. Louis series `BAMLH0A0HYM2`. Unless a cell specifies otherwise, the market analysis begins on January 1, 2025.


## 1. Environment and paths

Run this cell first. All generated figures are written to the project’s `outputs/` directory.


In [ ]:
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
import yfinance as yf


def find_project_root(start: Path) -> Path:
    """Locate the private-credit project from the repository root or notebook folder."""
    start = start.resolve()
    for candidate in (start, *start.parents):
        if candidate.name == "private-credit-article":
            return candidate
        nested = candidate / "private-credit-article"
        if nested.is_dir():
            return nested
    return start


PROJECT_ROOT = find_project_root(Path.cwd())
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Charts save to: {OUTPUT_DIR}")


## 2. Data and chart helpers

These functions download adjusted market prices, index series to 100, and save figures consistently.


In [ ]:
def download_prices(tickers, start="2025-01-01", end=None):
    data = yf.download(
        tickers,
        start=start,
        end=end,
        auto_adjust=True,
        progress=False
    )

    if data.empty:
        raise ValueError("No data downloaded. Check tickers, dates, or internet connection.")

    if isinstance(data.columns, pd.MultiIndex):
        prices = data["Close"]
    else:
        ticker_name = tickers if isinstance(tickers, str) else tickers[0]
        prices = data[["Close"]].rename(columns={"Close": ticker_name})

    prices = prices.dropna(axis=1, how="all")
    prices = prices.ffill()

    return prices


def index_prices(prices):
    indexed = prices.copy()

    for col in indexed.columns:
        first_valid = indexed[col].dropna().iloc[0]
        indexed[col] = indexed[col] / first_valid * 100

    return indexed


def calculate_drawdowns(prices):
    rolling_peak = prices.cummax()
    drawdowns = (prices / rolling_peak - 1) * 100
    return drawdowns


def save_chart(filename):
    path = OUTPUT_DIR / filename
    plt.savefig(path, dpi=300, bbox_inches="tight")
    print("Saved:", path)

## 3. BDCs versus public markets

Compare the BIZD BDC ETF with U.S. equities, high-yield bonds, leveraged loans, and regional banks. The shaded-period analysis isolates episodes when private-credit proxies diverged from broader risk markets.


In [ ]:
tickers = ["BIZD", "SPY", "HYG", "BKLN", "KRE"]

prices = download_prices(tickers, start="2025-01-01")
indexed = index_prices(prices)

plt.figure(figsize=(12, 6))

for ticker in indexed.columns:
    plt.plot(indexed.index, indexed[ticker], label=ticker)

plt.title("Indexed Performance: BDCs vs Public Markets")
plt.xlabel("Date")
plt.ylabel("Indexed Price (Start = 100)")
plt.legend()
plt.tight_layout()

save_chart("chart_1_bizd_vs_public_markets.png")
plt.show()

In [ ]:
# Define zones you want to highlight
zones = [
    {
        "start": "2025-03-01",
        "end": "2025-04-30",
        "label": "Broad risk-off",
        "note": "All risk assets sell off together"
    },
    {
        "start": "2025-09-01",
        "end": "2025-12-15",
        "label": "BDC divergence begins",
        "note": "BIZD weakens while public credit holds up"
    },
    {
        "start": "2026-01-01",
        "end": "2026-03-31",
        "label": "Repricing intensifies",
        "note": "BIZD falls to new lows"
    },
    {
        "start": "2026-04-01",
        "end": "2026-07-01",
        "label": "Risk assets recover",
        "note": "SPY/KRE rally; BIZD stays weak"
    }
]

plt.figure(figsize=(14, 7))

for ticker in indexed.columns:
    plt.plot(indexed.index, indexed[ticker], label=ticker, linewidth=2)

# Add shaded zones
for zone in zones:
    start = pd.to_datetime(zone["start"])
    end = pd.to_datetime(zone["end"])
    plt.axvspan(start, end, alpha=0.12)

    midpoint = start + (end - start) / 2
    plt.text(
        midpoint,
        indexed.max().max() + 1,
        zone["label"],
        ha="center",
        va="bottom",
        fontsize=9,
        rotation=0
    )

plt.title("Indexed Performance: BDCs vs Public Markets")
plt.xlabel("Date")
plt.ylabel("Indexed Price (Start = 100)")
plt.legend()
plt.tight_layout()

save_chart("chart_1_marked_zones.png")
plt.show()

In [ ]:
zone_results = []

for zone in zones:
    start = pd.to_datetime(zone["start"])
    end = pd.to_datetime(zone["end"])

    zone_data = indexed.loc[(indexed.index >= start) & (indexed.index <= end)]

    if len(zone_data) > 1:
        zone_return = (zone_data.iloc[-1] / zone_data.iloc[0] - 1) * 100

        for ticker, value in zone_return.items():
            zone_results.append({
                "Zone": zone["label"],
                "Ticker": ticker,
                "Return (%)": value
            })

zone_returns = pd.DataFrame(zone_results)

display(zone_returns.pivot(index="Zone", columns="Ticker", values="Return (%)").round(2))

In [ ]:
total_returns = (indexed.iloc[-1] / indexed.iloc[0] - 1) * 100
total_returns = total_returns.sort_values()

print("Total return over full period:")
display(total_returns.round(2))

In [ ]:
bizd_return = total_returns["BIZD"]

relative_gap = total_returns - bizd_return
relative_gap = relative_gap.sort_values(ascending=False)

print("How much each benchmark outperformed BIZD:")
display(relative_gap.round(2))

## 4. Dispersion across public BDCs

Measure performance differences across major listed BDCs and examine how those gaps changed during selected market regimes.


In [ ]:
bdc_tickers = ["ARCC", "OBDC", "FSK", "BXSL", "GBDC", "MAIN", "HTGC", "BCSF"]

bdc_prices = download_prices(bdc_tickers, start="2025-01-01")
bdc_indexed = index_prices(bdc_prices)

plt.figure(figsize=(12, 6))

for ticker in bdc_indexed.columns:
    plt.plot(bdc_indexed.index, bdc_indexed[ticker], label=ticker)

plt.title("Indexed Performance of Major Public BDCs")
plt.xlabel("Date")
plt.ylabel("Indexed Price (Start = 100)")
plt.legend(ncol=2)
plt.tight_layout()

save_chart("chart_2_individual_bdc_performance.png")
plt.show()

In [ ]:
zones = [
    {
        "start": "2025-03-01",
        "end": "2025-04-30",
        "label": "Broad BDC selloff"
    },
    {
        "start": "2025-09-01",
        "end": "2025-12-15",
        "label": "BDC dispersion begins"
    },
    {
        "start": "2026-01-01",
        "end": "2026-03-31",
        "label": "Repricing intensifies"
    },
    {
        "start": "2026-04-01",
        "end": "2026-07-01",
        "label": "Uneven recovery"
    }
]
plt.figure(figsize=(14, 7))

for ticker in bdc_indexed.columns:
    plt.plot(bdc_indexed.index, bdc_indexed[ticker], label=ticker, linewidth=2)

for zone in zones:
    start = pd.to_datetime(zone["start"])
    end = pd.to_datetime(zone["end"])
    plt.axvspan(start, end, alpha=0.10)

    midpoint = start + (end - start) / 2
    plt.text(
        midpoint,
        bdc_indexed.max().max() + 1,
        zone["label"],
        ha="center",
        va="bottom",
        fontsize=9
    )

plt.axhline(100, linestyle="--", linewidth=1)
plt.title("Individual BDC Performance With Market Zones")
plt.xlabel("Date")
plt.ylabel("Indexed Price (Start = 100)")
plt.legend(ncol=2)
plt.tight_layout()

save_chart("chart_2_marked_zones.png")
plt.show()

In [ ]:
bdc_total_returns = (bdc_prices.iloc[-1] / bdc_prices.iloc[0] - 1) * 100
bdc_total_returns = bdc_total_returns.sort_values()

print("Total return by BDC since start date:")
display(bdc_total_returns.round(2))

In [ ]:
bdc_total_returns_sorted = bdc_total_returns.sort_values()

plt.figure(figsize=(10, 6))
plt.barh(bdc_total_returns_sorted.index, bdc_total_returns_sorted.values)

plt.axvline(0, linestyle="--", linewidth=1)
plt.title("Total Return by Public BDC")
plt.xlabel("Total Return Since Start Date (%)")
plt.ylabel("BDC")
plt.tight_layout()

save_chart("chart_2_bdc_total_return_ranking.png")
plt.show()

In [ ]:
best_bdc = bdc_total_returns.idxmax()
worst_bdc = bdc_total_returns.idxmin()

best_return = bdc_total_returns.max()
worst_return = bdc_total_returns.min()
dispersion_gap = best_return - worst_return

print("Best performing BDC:", best_bdc, round(best_return, 2), "%")
print("Worst performing BDC:", worst_bdc, round(worst_return, 2), "%")
print("Performance gap:", round(dispersion_gap, 2), "percentage points")

In [ ]:
zone_results = []

for zone in zones:
    start = pd.to_datetime(zone["start"])
    end = pd.to_datetime(zone["end"])

    zone_data = bdc_indexed.loc[
        (bdc_indexed.index >= start) & 
        (bdc_indexed.index <= end)
    ]

    if len(zone_data) > 1:
        zone_return = (zone_data.iloc[-1] / zone_data.iloc[0] - 1) * 100

        for ticker, value in zone_return.items():
            zone_results.append({
                "Zone": zone["label"],
                "Ticker": ticker,
                "Return (%)": value
            })

bdc_zone_returns = pd.DataFrame(zone_results)

display(
    bdc_zone_returns
    .pivot(index="Zone", columns="Ticker", values="Return (%)")
    .round(2)
)

## 5. Drawdown comparison

Calculate each asset’s decline from its previous peak to distinguish broad risk-off periods from BDC-specific weakness.


In [ ]:
drawdown_tickers = ["BIZD", "SPY", "HYG", "BKLN", "KRE"]

drawdown_prices = download_prices(drawdown_tickers, start="2025-01-01")

rolling_peak = drawdown_prices.cummax()
drawdowns = (drawdown_prices / rolling_peak - 1) * 100

plt.figure(figsize=(12, 6))

for ticker in drawdowns.columns:
    plt.plot(drawdowns.index, drawdowns[ticker], label=ticker)

plt.title("Drawdowns: BDCs vs Broader Markets")
plt.xlabel("Date")
plt.ylabel("Drawdown from Previous Peak (%)")
plt.axhline(0, linestyle="--", linewidth=1)
plt.legend()
plt.tight_layout()

save_chart("chart_3_drawdowns.png")
plt.show()

In [ ]:
max_drawdowns = drawdowns.min().sort_values()

print("Maximum drawdown since start date:")
display(max_drawdowns.round(2))

## 6. BDC performance and high-yield spreads

Place BIZD beside the ICE BofA U.S. High Yield Index option-adjusted spread to compare public BDC pricing with conditions in traditional sub-investment-grade credit.


In [ ]:
# BIZD price data
bizd = download_prices("BIZD", start="2025-01-01")
bizd_indexed = index_prices(bizd)

# FRED high-yield spread data
fred_url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=BAMLH0A0HYM2"
hy_spread = pd.read_csv(fred_url)

hy_spread["observation_date"] = pd.to_datetime(hy_spread["observation_date"])
hy_spread["BAMLH0A0HYM2"] = pd.to_numeric(
    hy_spread["BAMLH0A0HYM2"],
    errors="coerce"
)

hy_spread = hy_spread.dropna()
hy_spread = hy_spread[hy_spread["observation_date"] >= "2025-01-01"]

fig, ax1 = plt.subplots(figsize=(12, 6))

ax1.plot(
    bizd_indexed.index,
    bizd_indexed["BIZD"],
    label="BIZD indexed"
)
ax1.set_xlabel("Date")
ax1.set_ylabel("BIZD Indexed Price, Start = 100")

ax2 = ax1.twinx()
ax2.plot(
    hy_spread["observation_date"],
    hy_spread["BAMLH0A0HYM2"],
    label="High-Yield OAS"
)
ax2.set_ylabel("High-Yield Spread (%)")

plt.title("BIZD vs High-Yield Credit Spreads")
fig.tight_layout()

save_chart("chart_4_bizd_vs_high_yield_spreads.png")
plt.show()

In [ ]:
print("BIZD start:", round(bizd_indexed["BIZD"].iloc[0], 2))
print("BIZD latest:", round(bizd_indexed["BIZD"].iloc[-1], 2))

print("High-yield spread start:", round(hy_spread["BAMLH0A0HYM2"].iloc[0], 2))
print("High-yield spread latest:", round(hy_spread["BAMLH0A0HYM2"].iloc[-1], 2))
print("High-yield spread max:", round(hy_spread["BAMLH0A0HYM2"].max(), 2))

## 7. Relative drawdown gaps

Measure how far BIZD’s drawdown diverged from high-yield bonds, leveraged loans, equities, and regional banks.


In [ ]:
# Assumes you already ran the Chart 3 code and have:
# drawdown_prices
# drawdowns

# If not, uncomment these two lines:
# drawdown_tickers = ["BIZD", "SPY", "HYG", "BKLN", "KRE"]
# drawdowns = (download_prices(drawdown_tickers, start="2025-01-01") / download_prices(drawdown_tickers, start="2025-01-01").cummax() - 1) * 100

gap_df = pd.DataFrame(index=drawdowns.index)

for benchmark in ["HYG", "BKLN", "SPY", "KRE"]:
    gap_df[benchmark] = drawdowns[benchmark] - drawdowns["BIZD"]

plt.figure(figsize=(14, 7))

for benchmark in gap_df.columns:
    # Make HYG and BKLN the main focus
    if benchmark in ["HYG", "BKLN"]:
        plt.plot(gap_df.index, gap_df[benchmark], label=f"{benchmark} vs BIZD", linewidth=3)
    else:
        plt.plot(gap_df.index, gap_df[benchmark], label=f"{benchmark} vs BIZD", linewidth=1.8, alpha=0.7)

    # Mark the maximum gap point
    max_gap_date = gap_df[benchmark].idxmax()
    max_gap_value = gap_df[benchmark].max()

    plt.scatter(max_gap_date, max_gap_value, s=70, zorder=5)
    plt.annotate(
        f"{benchmark}: {max_gap_value:.1f} pp",
        xy=(max_gap_date, max_gap_value),
        xytext=(10, 10),
        textcoords="offset points",
        fontsize=9
    )

plt.axhline(0, linestyle="--", linewidth=1)

plt.title("Drawdown Gap vs BIZD\nHow Much Less Severe Benchmark Drawdowns Were")
plt.xlabel("Date")
plt.ylabel("Drawdown Gap vs BIZD (Percentage Points)")
plt.legend()
plt.tight_layout()

save_chart("chart_3_drawdown_gap_all_in_one_with_markers.png")
plt.show()

## 8. Rolling volatility

Estimate 30-day annualized volatility to compare the intensity and persistence of price risk across the selected assets.


In [ ]:
vol_tickers = ["BIZD", "SPY", "HYG", "BKLN", "KRE"]

vol_prices = download_prices(vol_tickers, start="2025-01-01")
vol_returns = vol_prices.pct_change().dropna()

rolling_vol = vol_returns.rolling(30).std() * np.sqrt(252) * 100

plt.figure(figsize=(12, 6))

for ticker in rolling_vol.columns:
    plt.plot(rolling_vol.index, rolling_vol[ticker], label=ticker)

plt.title("30-Day Rolling Annualized Volatility")
plt.xlabel("Date")
plt.ylabel("Volatility (%)")
plt.legend()
plt.tight_layout()

save_chart("chart_5_rolling_volatility.png")
plt.show()

In [ ]:
latest_vol = rolling_vol.iloc[-1].sort_values(ascending=False)
avg_vol = rolling_vol.mean().sort_values(ascending=False)
max_vol = rolling_vol.max().sort_values(ascending=False)

vol_summary = pd.DataFrame({
    "Latest 30D Vol (%)": latest_vol,
    "Average 30D Vol (%)": avg_vol,
    "Max 30D Vol (%)": max_vol
})

display(vol_summary.round(2))

## 9. Relationships across BDC returns

Use correlations and single-factor regressions against MAIN to assess whether individual BDCs moved together or displayed company-specific behaviour.


In [ ]:
bdc_tickers = ["ARCC", "OBDC", "FSK", "BXSL", "GBDC", "MAIN", "HTGC", "BCSF"]

bdc_prices = download_prices(bdc_tickers, start="2025-01-01")
bdc_returns = bdc_prices.pct_change().dropna()

display(bdc_returns.head())

In [ ]:
main_returns = bdc_returns["MAIN"]

regression_results = []

for ticker in bdc_returns.columns:
    if ticker != "MAIN":
        y = bdc_returns[ticker]
        x = main_returns

        # Align the two return series
        regression_data = pd.concat([y, x], axis=1).dropna()
        regression_data.columns = [ticker, "MAIN"]

        y_reg = regression_data[ticker]
        x_reg = sm.add_constant(regression_data["MAIN"])

        model = sm.OLS(y_reg, x_reg).fit()

        correlation = regression_data[ticker].corr(regression_data["MAIN"])

        regression_results.append({
            "BDC": ticker,
            "Correlation with MAIN": correlation,
            "Beta to MAIN": model.params["MAIN"],
            "Alpha Daily (%)": model.params["const"] * 100,
            "R-squared": model.rsquared,
            "P-value": model.pvalues["MAIN"]
        })

main_regression_table = pd.DataFrame(regression_results)

main_regression_table = main_regression_table.sort_values(
    "Correlation with MAIN",
    ascending=False
)

display(main_regression_table.round(4))

## 10. FSK relative-value case study

Construct an equal-weight BDC peer basket excluding FSK, compare indexed performance, and measure FSK’s deviation and correlation with the peer group.


In [ ]:
# Make sure you already have bdc_prices and bdc_indexed from Chart 2
# If not, rerun this:
bdc_tickers = ["ARCC", "BCSF", "BXSL", "FSK", "GBDC", "HTGC", "MAIN", "OBDC"]

bdc_prices = download_prices(bdc_tickers, start="2025-01-01")
bdc_indexed = index_prices(bdc_prices)

# Define peer group excluding FSK
peer_tickers_ex_fsk = [ticker for ticker in bdc_indexed.columns if ticker != "FSK"]

# Equal-weight peer basket excluding FSK
peer_basket_ex_fsk = bdc_indexed[peer_tickers_ex_fsk].mean(axis=1)

# FSK deviation from peer basket
fsk_deviation = bdc_indexed["FSK"] - peer_basket_ex_fsk

display(peer_basket_ex_fsk.head())
display(fsk_deviation.head())

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(peer_basket_ex_fsk.index, peer_basket_ex_fsk, label="BDC peer basket excluding FSK", linewidth=3)
plt.plot(bdc_indexed.index, bdc_indexed["FSK"], label="FSK", linewidth=3)

plt.axhline(100, linestyle="--", linewidth=1)

plt.title("FSK vs BDC Peer Basket Excluding FSK")
plt.xlabel("Date")
plt.ylabel("Indexed Price (Start = 100)")
plt.legend()
plt.tight_layout()

save_chart("chart_2_fsk_vs_peer_basket_ex_fsk.png")
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(fsk_deviation.index, fsk_deviation, linewidth=3)

plt.axhline(0, linestyle="--", linewidth=1)

plt.title("FSK Deviation From BDC Peer Basket")
plt.xlabel("Date")
plt.ylabel("FSK Minus Peer Basket (Indexed Points)")
plt.tight_layout()

save_chart("chart_2_fsk_deviation_from_peer_basket.png")
plt.show()

In [ ]:
fsk_final_value = bdc_indexed["FSK"].iloc[-1]
peer_final_value = peer_basket_ex_fsk.iloc[-1]
final_deviation = fsk_deviation.iloc[-1]

max_negative_deviation = fsk_deviation.min()
max_negative_deviation_date = fsk_deviation.idxmin()

fsk_total_return = (bdc_prices["FSK"].iloc[-1] / bdc_prices["FSK"].iloc[0] - 1) * 100

peer_basket_price = bdc_prices[peer_tickers_ex_fsk].mean(axis=1)
peer_total_return = (peer_basket_ex_fsk.iloc[-1] / peer_basket_ex_fsk.iloc[0] - 1) * 100

print("FSK final indexed value:", round(fsk_final_value, 2))
print("Peer basket final indexed value:", round(peer_final_value, 2))
print("Final FSK deviation from peers:", round(final_deviation, 2), "indexed points")
print("Maximum negative FSK deviation:", round(max_negative_deviation, 2), "indexed points")
print("Date of maximum negative deviation:", max_negative_deviation_date.date())
print()
print("FSK total return:", round(fsk_total_return, 2), "%")
print("Peer basket total return:", round(peer_total_return, 2), "%")
print("FSK underperformance vs peer basket:", round(fsk_total_return - peer_total_return, 2), "percentage points")

In [ ]:
bdc_returns = bdc_prices.pct_change().dropna()

peer_returns_ex_fsk = bdc_returns[peer_tickers_ex_fsk]

# Average pairwise correlation among all non-FSK BDCs
peer_corr_matrix = peer_returns_ex_fsk.corr()

# Get only the off-diagonal correlations
peer_corr_values = []

for i in range(len(peer_corr_matrix.columns)):
    for j in range(i + 1, len(peer_corr_matrix.columns)):
        peer_corr_values.append(peer_corr_matrix.iloc[i, j])

average_peer_correlation = np.mean(peer_corr_values)

# FSK correlation with the peer basket
peer_basket_returns = peer_returns_ex_fsk.mean(axis=1)
fsk_returns = bdc_returns["FSK"]

fsk_corr_with_peer_basket = fsk_returns.corr(peer_basket_returns)

print("Average correlation among non-FSK BDCs:", round(average_peer_correlation, 3))
print("FSK correlation with non-FSK peer basket:", round(fsk_corr_with_peer_basket, 3))

## Limitations

This analysis uses public-market proxies for private credit. Listed BDC prices incorporate liquidity, sentiment, leverage, portfolio composition, management quality, and dividend expectations, so they should not be interpreted as direct marks on the entire private-credit market. Yahoo Finance data may also be revised, and rerunning the notebook later can change results because the download end date is not fixed.
